In [1]:
import matplotlib
matplotlib.use("WebAgg")
import pandas as pd
import numpy as np
import math as m
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt

In [2]:
# instellen van een webserver waar matplotlib de grafieken kan weergeven
plt.rcParams["webagg.address"] = "0.0.0.0"
plt.rcParams["webagg.port"] = 8988
plt.rcParams["webagg.open_in_browser"] = False

In [10]:
def df_load(meting):
    return pd.read_csv(f"datafiles/Basler_a2A2464-meting{meting}_11nov.mqa")

## load in all coordinates with P1 as the origin (lab frame)
def load_points(dataframe):
    df, points= dataframe, {}
    for i in [1,2,3]: points[i]= (df[f"Point #{i}.X"]-df["Point #1.X"], df[f"Point #{i}.Y"]-df["Point #1.Y"])
    return points
    
## load in all coordinates with P2 as the origin (translated frame)
def load_points_relative(dataframe):
    df, points_relative= dataframe, {}
    for i in [1,2,3]: points_relative[i]= (df[f"Point #{i}.X"]-df["Point #2.X"], df[f"Point #{i}.Y"]-df["Point #2.Y"])
    return points_relative

## load in all sizes (later usefull for errorbars)
def load_sizes(dataframe):
    df, sizes= dataframe, {}
    for i in [1,2,3]:
        sizes[i]= (df[f"Point #{i}.SizeX"], df[f"Point #{i}.SizeY"])
    return sizes


def load_time(dataframe):
    df= dataframe
    return df["Time"]

def load_frame(dataframe):
    df= dataframe
    return df["Frame"]


## load ~meting1~.mqa as df 
#df= df_load(1)

## load points as a library of tuples: 
# visually, points= {(x1:[..],y1:[..]),(x2:[..],y2:[..]),(x3:[..],y3:[..])}
# call on them using: points[1, 2 or 3][0 for x, 1 for y][element position]
#points= load_points()
#points_relative= load_points_relative()
#sizes= load_sizes()

In [ ]:
## create a class to for all calculations
class frame:
    
    def __init__(self, meting):
        self.df= df_load(meting)
        self.points= load_points(self.df)
        self.points_relative= load_points_relative(self.df)

        self.time= load_time(self.df)
        self.frame= load_frame(self.df)

        ## for easier coding 
        self.x1, self.y1= self.points_relative[1]
        self.x2, self.y2= self.points_relative[2]
        self.x3, self.y3= self.points_relative[3]

        ## lenght in mm and mass in gram
        #errors will be added later
        self.L1, self.L2= 225, 220
        #self.L1_err, self.L2_err= 
        self.m1, self.m2= 338, 418
        #self.m1_err, self.m2_err= 
        
    def angle_A(self):
        return np.arctan2(-self.x1, self.y1) * 180 / np.pi

    def angle_B(self):
        return np.arctan2(self.x3, -self.y3) * 180 / np.pi

    def dt(self):
        return np.diff(self.time)

    def dpsi(self, angle):
        return np.diff(getattr(self, f"angle_{angle}")())

    def angular_v(self, angle):
        theta = getattr(self, f"angle_{angle}")()
        return np.gradient(theta, self.time)

        

test= frame(1)
print(test.angular_v("A"))

[ 10.04240695   7.16174366  -1.35631111 ... -33.16362166 -20.31537138
   0.        ]
